In [ ]:
import os
import torch
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import open_clip

# ✅ 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

# ✅ 모델 로딩
model_name = "ViT-bigG-14"
pretrained = "laion2b_s39b_b160k"
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
tokenizer = open_clip.get_tokenizer(model_name)

# ✅ 데이터 로딩
test = pd.read_csv('./test.csv')
submission = pd.read_csv('./sample_submission.csv')

# ✅ 프롬프트 템플릿
prompt_templates = [
    lambda q, c: f"{c}",
    lambda q, c: f"The image most likely shows: {c}",
    lambda q, c: f"Based on the image, the answer is: {c}",
    lambda q, c: f"This is a photo of {c}.",
    lambda q, c: f"It looks like: {c}",
    lambda q, c: f"An image depicting {c}.",
    lambda q, c: f"The best description is {c}.",
    lambda q, c: f"Question: {q} Answer: {c}",
    lambda q, c: f"{q} — {c}",
    lambda q, c: f"Visual answer: {c}",
]

# ✅ 추론 루프
answers = []

for _, row in tqdm(test.iterrows(), total=len(test)):
    image = Image.open(row['img_path']).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    choices = [row[c] for c in ['A', 'B', 'C', 'D']]
    total_probs = torch.zeros(4).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        for template in prompt_templates:
            prompts = [template(row['Question'], c) for c in choices]
            text_tokens = tokenizer(prompts).to(device)
            text_features = model.encode_text(text_tokens)
            text_features /= text_features.norm(dim=-1, keepdim=True)

            similarity = (image_features @ text_features.T).squeeze(0)
            probs = torch.softmax(similarity * 10, dim=-1)
            total_probs += probs

    avg_probs = total_probs / len(prompt_templates)
    pred_idx = avg_probs.argmax().item()
    pred = chr(65 + pred_idx)
    answers.append(pred)

# ✅ 저장
submission['answer'] = answers
submission.to_csv('./final_submit_real.csv', index=False)
print("✅ 최종 제출 파일 저장 완료: final_submit.csv")
